# SemanticEmbed: Bring Your Own Graph

Load your own architecture and encode it. Supports JSON files, inline edge lists, and common data sources.

[GitHub](https://github.com/semanticembed/sdk) | [Website](https://semanticembed.com)


In [ ]:
!pip install -q semanticembed
from semanticembed import encode, encode_file, report
import json
print("Ready.")


---
## Option 1: Paste an Edge List

Simplest path. Define your edges as Python tuples.


In [ ]:
# ============================================================
# REPLACE THESE WITH YOUR OWN EDGES
# ============================================================
my_edges = [
    ("service-a", "service-b"),
    ("service-a", "service-c"),
    ("service-b", "service-d"),
    ("service-c", "service-d"),
    ("service-d", "service-e"),
]

result = encode(my_edges)
print(result.table)
print()
print(report(result))


---
## Option 2: Upload a JSON File

Upload a JSON file with this format:

```json
{
  "edges": [
    {"source": "A", "target": "B"},
    {"source": "B", "target": "C"}
  ]
}
```


In [ ]:
# Upload from your machine (Colab)
from google.colab import files

print("Select a JSON file with your edge list:")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    result = encode_file(filename)
    print(f"\nEncoded {filename}")
    print(f"Encoding time: {result.encoding_time_ms:.2f}ms")
    print()
    print(result.table)
    print()
    print(report(result))


---
## Option 3: Load a Sample Graph from the Repo


In [ ]:
import urllib.request

samples = {
    "Google Online Boutique": "examples/google_online_boutique.json",
    "Weaveworks Sock Shop": "examples/weaveworks_sock_shop.json",
    "Sample Data Pipeline": "examples/sample_pipeline.json",
}

base_url = "https://raw.githubusercontent.com/semanticembed/sdk/main/"

for name, path in samples.items():
    try:
        with urllib.request.urlopen(base_url + path) as resp:
            data = json.loads(resp.read())
        result = encode(data["edges"])
        print(f"\n{'=' * 60}")
        print(f"{name} ({result.graph_info['nodes']} nodes, {result.graph_info['edges']} edges)")
        print(f"{'=' * 60}")
        print(result.table)
        print()
        print(report(result))
    except Exception as e:
        print(f"Could not load {name}: {e}")
        print("(Expected if repo is not yet public. Use Options 1 or 2 instead.)")


---
## Option 4: From OpenTelemetry Traces

If you have OTel trace data, extract parent-child span relationships as edges:

```python
# Pseudocode -- adapt to your trace format
edges = []
for trace in traces:
    for span in trace.spans:
        if span.parent_id:
            parent_service = get_service(span.parent_id)
            child_service = span.service_name
            edges.append((parent_service, child_service))

# Deduplicate
edges = list(set(edges))
result = encode(edges)
```


---
## Option 5: From Kubernetes

Extract service dependencies from your cluster:

```bash
# Get service-to-service calls from network policies or observed traffic
kubectl get networkpolicies -o json | jq '...' > edges.json

# Or from Istio telemetry
istioctl dashboard kiali  # export the graph as JSON
```

Then load the JSON:

```python
result = encode_file("edges.json")
```


---
## Export Results

Save the encoding and risk report for later use.


In [ ]:
# Encode any graph
result = encode([
    ("A", "B"), ("A", "C"), ("B", "D"),
    ("C", "D"), ("D", "E"), ("D", "F"),
])

# Export as JSON
output = result.json()
with open("encoding_output.json", "w") as f:
    json.dump(output, f, indent=2)

print("Saved encoding_output.json")
print(json.dumps(output, indent=2))


---
## Free Tier Limit

The free tier supports graphs up to **50 nodes**. For larger production graphs, activate a license key:

```python
import semanticembed
semanticembed.license_key = "your-key-here"

# Now unlimited
result = encode(large_edges)
```

[Get a license key](https://semanticembed.com/pricing)

---

*Patent pending. Application #63/994,075.*
